<a href="https://colab.research.google.com/github/Asuskf/from-nlp-to-agents/blob/tokens/tokens/Tokenization%20Algorithms/Tokenization_Algorithms_Compare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Introduction:** The Tokenization Engine

Before a Large Language Model (LLM) can calculate probabilities and generate text, it must first transform human-readable information into a representation that can be processed mathematically: a sequence of discrete integer IDs. This notebook explores the algorithmic and mathematical foundations of four major tokenization paradigms. Specifically, it examines how modern language models convert raw data into tokens, covering approaches based on frequency-driven subword merging, probabilistic segmentation, unified multimodal representations, and direct byte-level encoding.


# 1. Byte-Level Byte Pair Encoding (BBPE)

## Description

Byte-Level Byte Pair Encoding (BBPE) adapts a classic data compression algorithm for tokenization. Rather than relying on a fixed vocabulary of words or subwords, BBPE begins with a vocabulary containing the **256 possible byte values (UTF-8 bytes)**. During training, the tokenizer repeatedly identifies the most frequent adjacent pair of tokens and replaces it with a newly created token.

Over many iterations, the vocabulary gradually expands while the tokenized representation becomes more compact.

## Mathematical Foundation (Frequency Maximization)

At each iteration, the tokenizer computes the frequency of every adjacent token pair $(x,y)$ in the training corpus. The optimal merge is the pair that maximizes the frequency count function $C$:

$$
(a,b)_{\text{opt}}
=
\arg\max_{(x,y)}
C(x,y)
$$

After selecting the optimal pair:

- A new token is added to the vocabulary:

$$
|V| \leftarrow |V| + 1
$$

- Every occurrence of the selected pair is replaced by the new token, reducing the total sequence length.


In [1]:
from pydantic import BaseModel, Field
from typing import List, Dict, Tuple

class BPEState(BaseModel):
    """Holds the current state of our tokenized sequence and vocabulary."""
    token_ids: List[int] = Field(
        ...,
        description="The current list of integer token IDs representing the text."
    )
    vocab_size: int = Field(
        default=256,
        description="Starts at 256 (base bytes). Grows with each merge."
    )

class BPE_Tokenizer(BaseModel):
    """Simulates the Byte-Level BPE training process."""

    def get_stats(self, token_ids: List[int]) -> Dict[Tuple[int, int], int]:
        """Finds all adjacent pairs of tokens and counts their frequencies."""
        counts = {}
        for pair in zip(token_ids, token_ids[1:]):
            counts[pair] = counts.get(pair, 0) + 1
        return counts

    def merge(self, token_ids: List[int], pair: Tuple[int, int], new_id: int) -> List[int]:
        """Replaces all occurrences of the most frequent 'pair' with 'new_id'."""
        new_token_ids = []
        i = 0
        while i < len(token_ids):
            # If we find the pair, merge them into the new token
            if i < len(token_ids) - 1 and token_ids[i] == pair[0] and token_ids[i+1] == pair[1]:
                new_token_ids.append(new_id)
                i += 2 # Skip the next token since it was merged
            else:
                new_token_ids.append(token_ids[i])
                i += 1
        return new_token_ids


if __name__ == "__main__":
    # 1. Our target Spanish text (Notice the repeated words/syllables)
    text = "el gato y el pato"

    # 2. Convert raw string into UTF-8 bytes (Base vocabulary of 256)
    raw_bytes = text.encode("utf-8")
    initial_tokens = list(raw_bytes)

    # 3. Initialize Tokenizer and State
    tokenizer = BPE_Tokenizer()
    state = BPEState(token_ids=initial_tokens)

    print(f"Original Text: '{text}'")
    print(f"Initial Bytes (t=0): {state.token_ids}")
    print(f"Initial Sequence Length: {len(state.token_ids)} tokens\n")
    print("-" * 50)

    # 4. Perform 3 iterations of BPE merges
    num_merges = 3

    for i in range(num_merges):
        # Step A: Count frequencies of adjacent pairs
        stats = tokenizer.get_stats(state.token_ids)
        if not stats:
            break

        # Step B: Find the most frequent pair
        best_pair = max(stats, key=stats.get)
        occurrences = stats[best_pair]

        # Step C: Create a new token ID for this pair
        new_token_id = state.vocab_size

        print(f"Merge Iteration {i + 1}:")
        print(f"  Most frequent pair: {best_pair} (appeared {occurrences} times)")
        print(f"  Merging {best_pair} -> New Token ID: {new_token_id}")

        # Step D: Apply the merge to our sequence
        state.token_ids = tokenizer.merge(state.token_ids, best_pair, new_token_id)
        state.vocab_size += 1

        print(f"  New Sequence: {state.token_ids}")
        print(f"  New Length: {len(state.token_ids)} tokens")
        print("-" * 50)

    print("\nFinal Result:")
    print(f"Vocabulary Size grew from 256 to {state.vocab_size}")
    print(f"Sequence compressed down to {len(state.token_ids)} tokens.")

Original Text: 'el gato y el pato'
Initial Bytes (t=0): [101, 108, 32, 103, 97, 116, 111, 32, 121, 32, 101, 108, 32, 112, 97, 116, 111]
Initial Sequence Length: 17 tokens

--------------------------------------------------
Merge Iteration 1:
  Most frequent pair: (101, 108) (appeared 2 times)
  Merging (101, 108) -> New Token ID: 256
  New Sequence: [256, 32, 103, 97, 116, 111, 32, 121, 32, 256, 32, 112, 97, 116, 111]
  New Length: 15 tokens
--------------------------------------------------
Merge Iteration 2:
  Most frequent pair: (256, 32) (appeared 2 times)
  Merging (256, 32) -> New Token ID: 257
  New Sequence: [257, 103, 97, 116, 111, 32, 121, 32, 257, 112, 97, 116, 111]
  New Length: 13 tokens
--------------------------------------------------
Merge Iteration 3:
  Most frequent pair: (97, 116) (appeared 2 times)
  Merging (97, 116) -> New Token ID: 258
  New Sequence: [257, 103, 258, 111, 32, 121, 32, 257, 112, 258, 111]
  New Length: 11 tokens
----------------------------------

# 2. SentencePiece and the Unigram Model

## Description

Unlike BBPE, which deterministically merges frequent token pairs, the SentencePiece Unigram model formulates tokenization as a probabilistic optimization problem.

The tokenizer begins with a predefined vocabulary where every token has an associated probability (or log-probability). Given an input string, multiple valid segmentations may exist. The tokenizer searches for the segmentation with the highest overall probability using the **Viterbi algorithm**.

## Mathematical Foundation (Viterbi Decoding)

Assuming token independence (the unigram assumption), the probability of a token sequence

$$
\mathbf{x}=(x_1,\dots,x_M)
$$

is computed by summing the log-probabilities of its constituent tokens.

The optimal segmentation is

$$
\mathbf{x}^*
=
\arg\max_{\mathbf{x}}
\sum_{i=1}^{M}
\log P(x_i)
$$

where:

- $P(x_i)$ is the learned probability assigned to token $x_i$.
- The Viterbi algorithm efficiently searches the space of possible segmentations.

In [2]:
from pydantic import BaseModel, Field
from typing import List, Dict
import math

class UnigramVocab(BaseModel):
    """Holds our vocabulary and the log-probabilities of each token."""
    tokens: Dict[str, float] = Field(
        ...,
        description="Dictionary mapping a token to its log-probability. (Closer to 0 is better)"
    )

class SentencePieceTokenizer(BaseModel):
    """Simulates SentencePiece Unigram tokenization using the Viterbi algorithm."""
    vocab: UnigramVocab

    def tokenize(self, text: str) -> List[str]:
        """Finds the most probable token segmentation for the input text."""
        N = len(text)

        # best_scores[i] holds the maximum log-probability for the substring text[0:i]
        # We initialize with negative infinity
        best_scores = [-float('inf')] * (N + 1)
        best_scores[0] = 0.0  # Base case: empty string has 0 log-prob

        # best_paths[i] remembers where the best split came from to reconstruct the tokens
        best_paths = [0] * (N + 1)

        # Viterbi Algorithm: Build the best score dynamically
        for i in range(1, N + 1):
            for j in range(i):
                substring = text[j:i]
                if substring in self.vocab.tokens:
                    # Score = best score up to 'j' + probability of the new token
                    score = best_scores[j] + self.vocab.tokens[substring]

                    if score > best_scores[i]:
                        best_scores[i] = score
                        best_paths[i] = j

        # If we couldn't find a valid segmentation
        if best_scores[N] == -float('inf'):
            return ["<unk>"]

        # Backtrack to extract the winning tokens
        tokens = []
        current_index = N
        while current_index > 0:
            previous_index = best_paths[current_index]
            tokens.append(text[previous_index:current_index])
            current_index = previous_index

        # Reverse the list since we backtracked from the end
        return tokens[::-1]


if __name__ == "__main__":
    # 1. Define our Unigram vocabulary with Log-Probabilities
    # Note: SentencePiece uses '_' to represent spaces.
    # Higher negative numbers mean lower probability.
    vocab_data = {
        "_el": -2.0,
        "_gato": -3.5,
        "_ga": -6.0,
        "to": -4.0,
        "el": -5.0,
        "_duerme": -4.5,
        "_duer": -7.0,
        "me": -5.0,
        "o": -8.0
    }

    model = SentencePieceTokenizer(vocab=UnigramVocab(tokens=vocab_data))

    # 2. Target text (SentencePiece treats spaces as part of the string)
    text = "_el_gato_duerme"

    print(f"Target String: '{text}'\n")
    print("-" * 50)

    # 3. Tokenize
    final_tokens = model.tokenize(text)

    # 4. Show the result
    print("Viterbi Algorithm Segmentation Result:")
    print(f"Tokens: {final_tokens}")

    # Calculate the total score of this segmentation
    total_score = sum(vocab_data[t] for t in final_tokens)
    print(f"Total Log-Probability Score: {total_score}")
    print("-" * 50)

    # 5. Show an alternative (worse) segmentation for comparison
    alt_tokens = ["_el", "_ga", "to", "_duerme"]
    alt_score = sum(vocab_data.get(t, -float('inf')) for t in alt_tokens)

    print("\nAlternative (Sub-optimal) Segmentation:")
    print(f"Tokens: {alt_tokens}")
    print(f"Total Log-Probability Score: {alt_score}")
    print("Notice how the sub-optimal path has a lower (more negative) score.")

Target String: '_el_gato_duerme'

--------------------------------------------------
Viterbi Algorithm Segmentation Result:
Tokens: ['_el', '_gato', '_duerme']
Total Log-Probability Score: -10.0
--------------------------------------------------

Alternative (Sub-optimal) Segmentation:
Tokens: ['_el', '_ga', 'to', '_duerme']
Total Log-Probability Score: -16.5
Notice how the sub-optimal path has a lower (more negative) score.


# 3. Unified Multimodal Tokenization

## Description

Modern native multimodal language models process text and images within a single token space rather than through completely separate neural networks.

A unified tokenizer converts text into text tokens and images into discrete image tokens (typically produced by a vector quantizer). Image tokens occupy a reserved range of token IDs (for example, IDs starting above **100,000**) to prevent collisions with textual tokens.

Both modalities are concatenated into one continuous sequence that is processed autoregressively by the same transformer.

## Mathematical Foundation (Unified Discrete Projection)

Let

- $E_{\text{text}}$ denote the text tokenizer.
- $E_{\text{img}}$ denote the image vector quantizer.

Given text $T$ and image $I$, the unified token sequence is

$$
S =
E_{\text{text}}(T)
\oplus
E_{\text{img}}(I)
$$

where $\oplus$ denotes sequence concatenation.

Thus,

$$
S=(t_1,t_2,\dots,t_N)
$$

Once converted into a unified sequence, the language model estimates the joint probability using the standard autoregressive factorization:

$$
P(S)
=
\prod_{i=1}^{N}
P(t_i \mid t_{<i})
$$

where $t_{<i}$ denotes every token preceding position $i$.

In [3]:
from pydantic import BaseModel, Field
from typing import List, Union, Dict

# ==========================================
# Input Data Models
# ==========================================
class TextData(BaseModel):
    """Represents a text input."""
    type: str = "text"
    content: str

class ImageData(BaseModel):
    """Represents an image input."""
    type: str = "image"
    description: str
    patches: int = Field(..., description="Number of patches this image is divided into")

# ==========================================
# Unified Tokenizer
# ==========================================
class MultimodalTokenizer(BaseModel):
    """Simulates a unified tokenizer that maps different modalities to a shared token space."""

    # Text Vocabulary (Standard 0-50,000 range)
    text_vocab: Dict[str, int] = Field(default_factory=dict)

    # Image tokens use a reserved block of IDs (e.g., 100,000 to 108,192)
    IMG_TOKEN_OFFSET: int = 100000

    # Special Tokens
    IMG_START: int = 99998
    IMG_END: int = 99999

    def tokenize_text(self, text_input: TextData) -> List[int]:
        """Simulates BBPE text tokenization."""
        words = text_input.content.lower().split()
        # Fallback to token ID 1 (<unk>) if word is not in vocab
        return [self.text_vocab.get(word, 1) for word in words]

    def tokenize_image(self, img_input: ImageData) -> List[int]:
        """
        Simulates a VQ-VAE discrete image tokenizer.
        Real models map visual patches to a specific codebook ID.
        Here we generate simulated deterministic IDs based on the image type.
        """
        tokens = [self.IMG_START]
        base_visual_concept = hash(img_input.description) % 1000

        for i in range(img_input.patches):
            # Simulating specific visual tokens (e.g., "sky blue patch", "fur patch")
            patch_token = self.IMG_TOKEN_OFFSET + base_visual_concept + (i % 10)
            tokens.append(patch_token)

        tokens.append(self.IMG_END)
        return tokens

    def process_prompt(self, prompt: List[Union[TextData, ImageData]]) -> List[int]:
        """Flattens a multimodal prompt into a single 1D sequence of token IDs."""
        unified_sequence = []
        for item in prompt:
            if isinstance(item, TextData):
                unified_sequence.extend(self.tokenize_text(item))
            elif isinstance(item, ImageData):
                unified_sequence.extend(self.tokenize_image(item))
        return unified_sequence


if __name__ == "__main__":
    # 1. Initialize Tokenizer with a simulated Spanish text vocabulary
    mock_vocab = {
        "mira": 405,
        "este": 812,
        "paisaje:": 2099,
        "es": 310,
        "hermoso,": 5402,
        "¿verdad?": 8810
    }
    tokenizer = MultimodalTokenizer(text_vocab=mock_vocab)

    # 2. Construct a Multimodal Prompt
    # User says: "Mira este paisaje:" -> [Uploads Image of Mountains] -> "Es hermoso, ¿verdad?"
    multimodal_prompt = [
        TextData(content="Mira este paisaje:"),
        ImageData(description="montaña_nevada", patches=6), # Simulating a 2x3 image grid
        TextData(content="Es hermoso, ¿verdad?")
    ]

    print("Processing Multimodal Prompt...\n")
    print("-" * 60)

    # 3. Process the prompt
    final_token_sequence = tokenizer.process_prompt(multimodal_prompt)

    # 4. Display the step-by-step breakdown
    current_idx = 0
    for item in multimodal_prompt:
        if isinstance(item, TextData):
            tokens = tokenizer.tokenize_text(item)
            print(f"[TEXT IN]  '{item.content}'")
            print(f"[TOKENS]   {tokens}\n")
        elif isinstance(item, ImageData):
            tokens = tokenizer.tokenize_image(item)
            print(f"[IMG IN]   <Image: {item.description} | {item.patches} patches>")
            print(f"[TOKENS]   {tokens}  <-- Notice the >100,000 IDs + Start/End tags\n")

    print("-" * 60)
    print("Final Unified 1D Token Sequence (What the LLM actually sees):")
    print(final_token_sequence)

Processing Multimodal Prompt...

------------------------------------------------------------
[TEXT IN]  'Mira este paisaje:'
[TOKENS]   [405, 812, 2099]

[IMG IN]   <Image: montaña_nevada | 6 patches>
[TOKENS]   [99998, 100439, 100440, 100441, 100442, 100443, 100444, 99999]  <-- Notice the >100,000 IDs + Start/End tags

[TEXT IN]  'Es hermoso, ¿verdad?'
[TOKENS]   [310, 5402, 8810]

------------------------------------------------------------
Final Unified 1D Token Sequence (What the LLM actually sees):
[405, 812, 2099, 99998, 100439, 100440, 100441, 100442, 100443, 100444, 99999, 310, 5402, 8810]


# 4. The Token-Free (Byte-to-Byte) Paradigm

## Description

The Token-Free approach removes tokenizer training entirely.

Instead of learning subwords or word pieces, every input string is encoded directly as its UTF-8 byte representation. Consequently, the vocabulary remains permanently fixed at exactly **256 symbols** corresponding to the possible byte values.

This shifts complexity away from the tokenizer and toward the neural network itself, which must learn linguistic structure directly from byte sequences.

## Mathematical Foundation (Byte-Level Conditional Probability)

A text sequence is deterministically mapped into a byte sequence

$$
B=(b_1,b_2,\dots,b_K),
$$

where

$$
b_i \in \{0,1,\dots,255\}.
$$

The language model estimates

$$
P(B)
=
\prod_{k=1}^{K}
P(b_k \mid b_1,b_2,\dots,b_{k-1})
$$

Since the vocabulary size remains fixed,

$$
|V|=256.
$$

Although this greatly simplifies the tokenization process, it significantly increases sequence length. For example, a single Unicode emoji may require up to four UTF-8 bytes, meaning the model must process substantially longer sequences and learn long-range dependencies directly within the transformer.

In [4]:
from pydantic import BaseModel, Field
from typing import List

class ByteSequence(BaseModel):
    """Holds a sequence of raw bytes."""
    byte_ids: List[int] = Field(
        ...,
        description="A list of integers, strictly ranging from 0 to 255."
    )

    # Custom validator to ensure true byte-level constraints
    def validate_bytes(self):
        if any(b < 0 or b > 255 for b in self.byte_ids):
            raise ValueError("Invalid byte: All IDs must be between 0 and 255.")

class TokenFreeEngine(BaseModel):
    """A pure Byte-to-Byte engine (No training required)."""

    vocab_size: int = Field(
        default=256,
        frozen=True, # The vocabulary size never changes
        description="Strictly 256 possible bytes."
    )

    def encode(self, text: str) -> ByteSequence:
        """Converts text directly into UTF-8 bytes."""
        # text.encode("utf-8") returns a bytes object, list() converts it to ints
        raw_ints = list(text.encode("utf-8"))
        sequence = ByteSequence(byte_ids=raw_ints)
        sequence.validate_bytes()
        return sequence

    def decode(self, sequence: ByteSequence) -> str:
        """Decodes raw bytes back into human-readable text."""
        # Convert list of ints back to bytes, then decode.
        # errors="replace" handles cases where the LLM might predict an invalid UTF-8 sequence
        raw_bytes = bytes(sequence.byte_ids)
        return raw_bytes.decode("utf-8", errors="replace")


if __name__ == "__main__":
    # 1. Initialize our token-free engine
    engine = TokenFreeEngine()

    # 2. Target text (Includes standard ASCII, a special Spanish character, and an Emoji)
    text = "¡Hola 🌎!"

    print(f"Original Text: '{text}'")
    print(f"Fixed Vocabulary Size: {engine.vocab_size}\n")
    print("-" * 50)

    # 3. Encode the text directly to bytes
    encoded_seq = engine.encode(text)

    print("Byte-Level Encoding:")
    print(f"Tokens (Integers 0-255): {encoded_seq.byte_ids}")
    print(f"Sequence Length: {len(encoded_seq.byte_ids)} tokens\n")
    print("-" * 50)

    # 4. Let's analyze exactly how UTF-8 mapped this string
    print("Detailed Breakdown of the Bytes:")

    # We will decode character by character to show the byte cost
    for char in text:
        char_bytes = list(char.encode("utf-8"))
        print(f"  Character '{char}' -> Requires {len(char_bytes)} byte(s): {char_bytes}")

    print("-" * 50)

    # 5. Decode back to verify
    decoded_text = engine.decode(encoded_seq)
    print(f"Decoded back to text: '{decoded_text}'")

Original Text: '¡Hola 🌎!'
Fixed Vocabulary Size: 256

--------------------------------------------------
Byte-Level Encoding:
Tokens (Integers 0-255): [194, 161, 72, 111, 108, 97, 32, 240, 159, 140, 142, 33]
Sequence Length: 12 tokens

--------------------------------------------------
Detailed Breakdown of the Bytes:
  Character '¡' -> Requires 2 byte(s): [194, 161]
  Character 'H' -> Requires 1 byte(s): [72]
  Character 'o' -> Requires 1 byte(s): [111]
  Character 'l' -> Requires 1 byte(s): [108]
  Character 'a' -> Requires 1 byte(s): [97]
  Character ' ' -> Requires 1 byte(s): [32]
  Character '🌎' -> Requires 4 byte(s): [240, 159, 140, 142]
  Character '!' -> Requires 1 byte(s): [33]
--------------------------------------------------
Decoded back to text: '¡Hola 🌎!'
